In [1]:
import pandas as pd
import numpy as np

data_description = '/content/data/tanzanian-data/Data Description.pdf'
data_labels = '/content/data/tanzanian-data/DataLabels.csv'
df = pd.read_csv('/content/data/tanzanian-data/WaterPumps.csv')


# Dataset Description: Tanzanian Water Pumps

### Overview & Quantity of Data
The dataset used for this project is sourced from Taarifa and the Tanzanian Ministry of Water. It aims to predict which water pumps are functional, non-functional, or functional but in need of repair based on various characteristics of the waterpoint.

* **Number of Instance Rows:** 59,400
* **Number of Attribute Columns:** 40 features + 1 target variable (`status_group`)

### Target Variable
The target variable is **`status_group`** (categorical/nominal class label) with three classes:
1. **functional:** 32,259 instances (54.31%)
2. **non functional:** 22,824 instances (38.42%)
3. **functional needs repair:** 4,317 instances (7.27%)

---

### Summary Table of Attributes

| Attribute | Data Type | Attribute Type | Description | Potential Quality Issues / Missing % |
| :--- | :--- | :--- | :--- | :--- |
| **id** | int64 | Discrete (Identifier) | Unique identifier for each waterpoint. | None (0.0% missing) |
| **amount_tsh** | float64 | Continuous | Total static head (amount of water available to waterpoint). | Features many 0 values which may represent missing data. |
| **date_recorded**| object | Discrete (Date) | The date the data entry was entered. | Format consistency needs verification. |
| **funder** | object | Nominal | Who funded the well. | **6.12% missing** (3,637 values); misspelled duplicates. |
| **gps_height** | int64 | Continuous | Altitude of the well. | Contains negative values or zeros which could indicate noise. |
| **installer** | object | Nominal | Organization that installed the well. | **6.15% missing** (3,655 values); spelling mistakes. |
| **longitude** | float64 | Continuous | GPS longitude coordinate. | Contains zero values which are invalid for Tanzania. |
| **latitude** | float64 | Continuous | GPS latitude coordinate. | Near-zero values (placeholder values representing noise). |
| **basin** | object | Nominal | Geographic water basin. | None (0.0% missing). |
| **subvillage** | object | Nominal | Geographic subvillage location. | **0.62% missing** (371 values); high cardinality (19,287 values). |
| **region_code** | int64 | Nominal (Encoded) | Geographic region code. | None (0.0% missing). |
| **district_code**| int64 | Nominal (Encoded) | Geographic district code. | None (0.0% missing). |
| **extraction_type**| object | Nominal | The kind of extraction extraction the pump uses. | High redundancy with other extraction columns. |
| **management** | object | Nominal | How the waterpoint is managed. | None (0.0% missing). |
| **payment_type** | object | Nominal | What the water cost is. | None (0.0% missing). |
| **water_quality**| object | Nominal | Quality of the water. | None (0.0% missing). |
| **quantity** | object | Nominal | The quantity of water at the well. | None (0.0% missing). |
| **source** | object | Nominal | The source of the water. | None (0.0% missing). |
| **waterpoint_type**| object| Nominal | The type of waterpoint. | None (0.0% missing). |
| **status_group** | object | Nominal (Target) | Classification: Functional, Non-functional, or Needs Repair. | Imbalanced classes (7.27% in 'needs repair'). |

---

### Potential Data Quality Issues
1. **Missing Data:** Key categorical identifiers such as `funder` (6.12% missing), `installer` (6.15% missing), and `subvillage` (0.62% missing) contain null values.
2. **Invalid Placeholder Values (Noise):** Columns such as `longitude`, `latitude`, `gps_height`, and `amount_tsh` contain zeros that are highly likely to be structural placeholders rather than true values (since Tanzania's coordinates do not sit at zero latitude or longitude).
3. **High Cardinality & Redundant Encoding:** Categorical variables such as `funder`, `installer`, and `wpt_name` have thousands of distinct values, often with structural spelling inconsistencies (e.g., typos in names) that will need mapping or binning. Additionally, several categories like `extraction_type`, `extraction_type_group`, and `extraction_type_class` represent the same raw information grouped at different granularities.

In [2]:
import pandas as pd
import numpy as np

# Load target labels to understand the target variable
labels_df = pd.read_csv('/content/data/tanzanian-data/DataLabels.csv')

# Merge dataset with labels on 'id' to check distribution and alignment
full_df = pd.merge(df, labels_df, on='id')

print(f"WaterPumps shape: {df.shape}")
print(f"Labels shape: {labels_df.shape}")
print(f"Merged shape: {full_df.shape}\n")

# Analyze data types and missing values
info_df = pd.DataFrame({
    'Data Type': full_df.dtypes,
    'Missing Values': full_df.isnull().sum(),
    'Percentage Missing (%)': (full_df.isnull().sum() / len(full_df) * 100).round(2),
    'Unique Values': full_df.nunique()
})

print("=== Column Summary (First 15 Columns) ===")
print(info_df.head(15))

print("\n=== Target Variable Distribution ===")
if 'status_group' in full_df.columns:
    print(full_df['status_group'].value_value_counts(dropna=False) if hasattr(full_df['status_group'], 'value_value_counts') else full_df['status_group'].value_counts(dropna=False))
else:
    # fallback to find the likely target column
    possible_targets = [col for col in labels_df.columns if col != 'id']
    for col in possible_targets:
        print(f"{col} distribution:")
        print(full_df[col].value_counts(dropna=False))


WaterPumps shape: (59400, 40)
Labels shape: (59400, 2)
Merged shape: (59400, 41)

=== Column Summary (First 15 Columns) ===
              Data Type  Missing Values  Percentage Missing (%)  Unique Values
id                int64               0                    0.00          59400
amount_tsh      float64               0                    0.00             98
date_recorded    object               0                    0.00            356
funder           object            3637                    6.12           1896
gps_height        int64               0                    0.00           2428
installer        object            3655                    6.15           2145
longitude       float64               0                    0.00          57516
latitude        float64               0                    0.00          57517
wpt_name         object               2                    0.00          37399
num_private       int64               0                    0.00             65
basin  